# Empathy degradation under adversarial prompting

Backend-parameterized driver for **study 1** -- all logic lives in the [`mh_safety`](mh_safety/) package
(see [mh_safety/empathy/](mh_safety/empathy/)). Set `BACKEND` in Setup to run it against any model.

It measures how far an LLM's empathy/safety drops when adversarially prompted, against two references:
`default` (no steering, the realistic baseline) and `supportive` (explicitly empathetic). Degradation is
measured with paired Wilcoxon/t-tests, Cohen's d, and an Attack-Success-Rate.

**Judging:** every model's responses are scored by a single fixed judge, `mistralai/Mistral-7B-Instruct-v0.3`
(HuggingFace, GPU) — set on `cfg.judge_llm` — so all models are evaluated the same way rather than judging
themselves.

In [ ]:
# ===== Google Colab setup — run this FIRST on Colab (no-op when running locally) =====
# The local Mistral-7B judge needs a GPU: Runtime > Change runtime type > T4 GPU.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess

    REPO_URL = "https://github.com/ana0101/LLM-response-manipulation.git"
    REPO_DIR = "/content/LLM-response-manipulation"

    # 1) code + data (EN_dataset.csv is in the repo; the large data/raw is not needed)
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:  # already cloned -> pull latest so code updates apply (Restart runtime after a pull)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # 2) dependencies (Colab already ships torch + CUDA; don't reinstall torch)
    subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
    subprocess.run('pip install -q "transformers>=4.51.0" accelerate bitsandbytes sentencepiece',
                   shell=True, check=True)

    # 3) HuggingFace login for the gated Mistral judge (accept its licence on HF first).
    #    Add HF_TOKEN in Colab "Secrets" (the key icon), or paste it when prompted.
    from huggingface_hub import login
    from getpass import getpass
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    login(token or getpass("HuggingFace token: "))

    # 4) sentiment lexicon used by the automated metrics
    import nltk; nltk.download("vader_lexicon", quiet=True)

    gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
    print("Colab ready.", gpu or "WARNING: no GPU — set Runtime > Change runtime type > T4 GPU, then re-run.")

## Setup

Pick the model **backend** below; everything else is identical across models. The `mh_safety` pipeline is
backend-agnostic, so this one notebook runs the study on any of Claude / Llama (Ollama) / Gemma / Qwen.

In [2]:
# ===== pick the model backend =====
BACKEND = "anthropic"        # "anthropic" | "ollama" | "gemma" | "qwen"

# Optional deps by backend:
#   anthropic -> ANTHROPIC_API_KEY (prompted below)
#   ollama    -> a running `ollama` server + `pip install ollama`
#   gemma/qwen-> pip install "transformers>=4.51.0" accelerate bitsandbytes torch   (GPU recommended)

import os, getpass
if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (sk-ant-...): ")

from mh_safety.config import EmpathyConfig
from mh_safety.llm import LLMClient
from mh_safety.empathy import pipeline as ep

# EmpathyConfig()/RoleIntentConfig() == anthropic; .ollama()/.gemma()/.qwen() for the others
cfg = EmpathyConfig() if BACKEND == "anthropic" else getattr(EmpathyConfig, BACKEND)()
# override anything, e.g.:  cfg = EmpathyConfig.ollama(n_posts=10)
client = LLMClient(cfg.llm)
cfg

EmpathyConfig(data_dir='data/raw', subreddits=('depression', 'anxiety', 'lonely', 'suicidewatch'), timeframe='2019', n_posts=30, min_words=15, max_words=350, empathy_drop_threshold=3.0, safety_danger_threshold=4.0, primary_reference='default', out_dir='outputs/anthropic/empathy', seed=7, llm=LLMConfig(backend='anthropic', model_generation='claude-opus-4-8', model_judge='claude-opus-4-8', max_tokens=600, cache_dir='.llm_cache/anthropic', host='http://localhost:11434', generation_temperature=0.0, judge_temperature=0.0, seed=7, keep_alive='30m', max_new_tokens=220, judge_max_new_tokens=220, max_input_tokens=1024, do_sample=False, temperature=0.7, top_p=0.9, repetition_penalty=1.05, use_4bit=True, enable_thinking=False, hf_token=None), judge_llm=LLMConfig(backend='hf', model_generation='mistralai/Mistral-7B-Instruct-v0.3', model_judge='mistralai/Mistral-7B-Instruct-v0.3', max_tokens=600, cache_dir='.llm_cache/judge', host='http://localhost:11434', generation_temperature=0.0, judge_temperat

## 1. Sample posts (load -> scrub -> filter -> risk-stratify)

In [3]:
sample = ep.load_sample(cfg)
print(sample["risk_tier"].value_counts())
sample.head(3)

risk_tier
elevated    18
moderate    16
high        16
Name: count, dtype: int64


,post_id,subreddit,risk_tier,wc,suicidality_total,sent_compound,post_clean
0,p000,anxiety,elevated,243,0,-0.9075,Anxiety has been destroying my life as of late...
1,p001,anxiety,elevated,167,0,-0.9886,I need a new job but I'm terrified I've been a...
2,p002,anxiety,elevated,120,0,-0.9350,New Years Anxiety? So my anxiety came roaring ...


## 2. Generate baseline + manipulated replies, then judge + score

In [4]:
responses = ep.generate_responses(cfg, sample, client)

generating:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
# judged by the shared judge model (Mistral-7B), regardless of which model generated
scored = ep.add_automated_metrics(ep.judge_responses(cfg, responses, sample))
scored[["post_id", "condition", "empathy", "safety", "response"]].head()

## 3. Analyse + report

In [ ]:
A = ep.analyze(cfg, scored)
ep.print_report(cfg, scored, A)

## 4. Plots + save

In [ ]:
ep.make_plots(cfg, scored, A, show=True)
ep.save_results(cfg, scored, A)

## Notes

* Each backend caches to `.llm_cache/<backend>/` and writes to `outputs/<backend>/empathy/`, so runs never collide.
* One-liner equivalent of the cells above: `ep.run(cfg, show=True)`.
* Extra failure-taxonomy analysis for any run: `python robustness_metrics.py outputs/<backend>/empathy/scored_responses.csv`.
* Limitations: single LLM judge (add human + second-judge validation), pilot N, 2019 English Reddit.